# Sentiment Analysis of Tweets using TF-IDF and Visualisation of the Results

**Aim:** To perform sentiment analysis on a dataset of tweets using TF-IDF and a classification model, and to
visualise the sentiment results.

**Dataset:** Sentiment140 (1.6 million tweets, 800,000 positive and 800,000 negative).

**Steps:** clean tweets → TF-IDF features → train Logistic Regression → evaluate → predict new posts →
**test** (model comparison, ROC, confidence analysis, cleaning unit tests, custom tweets, error analysis).

In [ ]:
!pip install -q kagglehub

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix
)

import kagglehub

## 1. Download and load the dataset

In [ ]:
# [LOAD]
path = kagglehub.dataset_download("kazanova/sentiment140")
print("Dataset path:", path)
print(os.listdir(path))

file_path = os.path.join(path, "training.1600000.processed.noemoticon.csv")

df = pd.read_csv(file_path, encoding="latin-1", header=None)
print("Dataset shape:", df.shape)

df.columns = ["sentiment", "id", "date", "query", "username", "tweet"]
display(df.head())

In [ ]:
# 0 = negative, 4 = positive  ->  0 / 1
df["sentiment"] = df["sentiment"].replace({0: 0, 4: 1})

print(df["sentiment"].value_counts())

In [ ]:
sentiment_counts = df["sentiment"].value_counts()

plt.figure(figsize=(7, 5))
plt.bar(["Negative", "Positive"], [sentiment_counts.get(0, 0), sentiment_counts.get(1, 0)])
plt.xlabel("Sentiment")
plt.ylabel("Number of Tweets")
plt.title("Sentiment Distribution")
plt.show()

## 2. Clean the tweets

In [ ]:
def clean_tweet(text):
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove mentions
    text = re.sub(r"@\w+", "", text)

    # Remove hashtag symbol but keep the word
    text = re.sub(r"#", "", text)

    # Remove RT
    text = re.sub(r"\brt\b", "", text)

    # Remove punctuation and special characters
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["clean_tweet"] = df["tweet"].apply(clean_tweet)

display(df[["tweet", "clean_tweet"]].head(10))

In [ ]:
# Remove tweets that became empty after cleaning
df = df[df["clean_tweet"].str.len() > 0]
print("Dataset after cleaning:", df.shape)

X = df["clean_tweet"]
y = df["sentiment"]

print("\nFeatures:")
print(X.head())
print("\nLabels:")
print(y.head())

## 3. Train / test split and TF-IDF

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

In [ ]:
tfidf = TfidfVectorizer(
    max_features=50000,
    stop_words="english",
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape :", X_test_tfidf.shape)

feature_names = tfidf.get_feature_names_out()
print("\nNumber of TF-IDF features:", len(feature_names))
print("First 50 features:")
print(feature_names[:50])

## 4. Train the classification model

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

print("Model training completed.")

In [ ]:
y_pred = model.predict(X_test_tfidf)

print("Predictions generated.")
print("\nFirst 20 predictions:")
print(y_pred[:20])
print("\nActual labels:")
print(y_test.iloc[:20].values)

## 5. Evaluate the model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("========== MODEL PERFORMANCE ==========")
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1-score  : {f1 * 100:.2f}%")

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

cm = confusion_matrix(y_test, y_pred)
print(cm)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["Negative", "Positive"],
            yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")
plt.title("Confusion Matrix")
plt.show()

## 6. Visualise the predicted sentiment

In [ ]:
predicted_counts = pd.Series(y_pred).value_counts()

plt.figure(figsize=(7, 5))
plt.bar(["Negative", "Positive"], [predicted_counts.get(0, 0), predicted_counts.get(1, 0)])
plt.xlabel("Predicted Sentiment")
plt.ylabel("Number of Tweets")
plt.title("Predicted Sentiment Distribution")
plt.show()

## 7. Predict the sentiment of new posts

In [ ]:
new_tweets = [
    "I absolutely love this product, it is amazing!",
    "This is the worst experience I have ever had.",
    "The product is excellent and works perfectly.",
    "I am very disappointed and unhappy with this."
]

# Clean tweets
clean_new_tweets = [clean_tweet(tweet) for tweet in new_tweets]

# Convert to TF-IDF
new_tweets_tfidf = tfidf.transform(clean_new_tweets)

# Predict
predictions = model.predict(new_tweets_tfidf)
probabilities = model.predict_proba(new_tweets_tfidf)

print("========== SENTIMENT PREDICTIONS ==========\n")
for tweet, prediction, probability in zip(new_tweets, predictions, probabilities):
    sentiment = "Positive" if prediction == 1 else "Negative"
    confidence = max(probability) * 100
    print("Tweet     :", tweet)
    print("Sentiment :", sentiment)
    print(f"Confidence: {confidence:.2f}%")
    print()

---
# Testing the sentiment model

1. **Unit tests of the tweet-cleaning function.**
2. **Model comparison** (Logistic Regression, Linear SVM, Naive Bayes) with ROC curves.
3. **Confidence analysis** - is the model right more often when it is more confident?
4. **Custom tweets** with a known correct answer, including difficult cases.
5. **Most informative words** and **error analysis**.

## Test 1: unit tests of the cleaning function

In [ ]:
cleaning_tests = [
    ("Lowercase",
     "I LOVE It", "i love it"),
    ("URL removed",
     "check this http://twitpic.com/2y1zl now", "check this now"),
    ("www link removed",
     "visit www.example.com today", "visit today"),
    ("Mention removed",
     "@switchfoot thats a bummer", "thats a bummer"),
    ("Hashtag symbol removed, word kept",
     "so happy #blessed", "so happy blessed"),
    ("RT removed",
     "RT great news", "great news"),
    ("Punctuation and digits removed",
     "wow!!! 100% sure :)", "wow sure"),
    ("Extra spaces removed",
     "too    many   spaces", "too many spaces"),
    ("Tweet with only a mention becomes empty",
     "@someone", ""),
]

rows = []
for name, raw, expected in cleaning_tests:
    actual = clean_tweet(raw)
    rows.append({"Test": name, "Input": raw, "Expected": expected, "Actual": actual,
                 "Result": "PASS" if actual == expected else "FAIL"})

unit_df = pd.DataFrame(rows)
display(unit_df)
print(f"Cleaning tests passed: {(unit_df['Result'] == 'PASS').sum()} / {len(unit_df)}")

## Test 2: compare three models and plot ROC curves

In [ ]:
svm_model = LinearSVC(max_iter=5000)
svm_model.fit(X_train_tfidf, y_train)

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

pred_by_model = {
    "Logistic Regression": y_pred,
    "Linear SVM": svm_model.predict(X_test_tfidf),
    "Naive Bayes": nb_model.predict(X_test_tfidf),
}
score_by_model = {
    "Logistic Regression": model.predict_proba(X_test_tfidf)[:, 1],
    "Linear SVM": svm_model.decision_function(X_test_tfidf),
    "Naive Bayes": nb_model.predict_proba(X_test_tfidf)[:, 1],
}

rows = []
for name, pred in pred_by_model.items():
    rows.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, pred), 4),
        "Precision": round(precision_score(y_test, pred), 4),
        "Recall": round(recall_score(y_test, pred), 4),
        "F1-score": round(f1_score(y_test, pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, score_by_model[name]), 4),
    })
comparison = pd.DataFrame(rows).set_index("Model")
display(comparison)
print("Best model by F1-score:", comparison["F1-score"].idxmax())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, s in score_by_model.items():
    fpr, tpr, _ = roc_curve(y_test, s)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, s):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", label="Random guess")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curves")
axes[0].legend(loc="lower right")

cm_norm = confusion_matrix(y_test, y_pred, normalize="true") * 100
sns.heatmap(cm_norm, annot=True, fmt=".1f", cmap="Blues",
            xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"], ax=axes[1])
axes[1].set_title("Confusion matrix of Logistic Regression (% of each actual class)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
plt.tight_layout()
plt.show()

## Test 3: is the model more accurate when it is more confident?

In [ ]:
proba_test = model.predict_proba(X_test_tfidf)
confidence = proba_test.max(axis=1)
correct = (y_pred == y_test.values)

bins = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0001]
labels = ["50-60%", "60-70%", "70-80%", "80-90%", "90-100%"]
conf_bin = pd.cut(confidence, bins=bins, labels=labels, right=False)

conf_df = (pd.DataFrame({"Confidence": conf_bin, "Correct": correct})
           .groupby("Confidence", observed=True)
           .agg(Tweets=("Correct", "size"), Accuracy=("Correct", "mean")))
conf_df["Share of tweets"] = (conf_df["Tweets"] / conf_df["Tweets"].sum() * 100).round(1).astype(str) + " %"
conf_df["Accuracy"] = (conf_df["Accuracy"] * 100).round(1)
display(conf_df)

plt.figure(figsize=(8, 4.5))
plt.bar(conf_df.index.astype(str), conf_df["Accuracy"])
plt.axhline(accuracy * 100, color="red", linestyle="--", label=f"Overall accuracy {accuracy * 100:.1f}%")
plt.ylim(40, 100)
plt.xlabel("Model confidence")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy by confidence level")
plt.legend()
plt.show()

## Test 4: custom tweets with a known correct answer

In [ ]:
# (tweet, correct label)  1 = positive, 0 = negative
custom_tests = [
    ("I absolutely love this phone, the battery lasts forever! #happy", 1),
    ("What a beautiful day, feeling so grateful and excited for the weekend", 1),
    ("Just got the job!!! Best news ever, thank you all so much", 1),
    ("Amazing concert last night, the band was fantastic http://pic.example.com/x1", 1),
    ("@airline you lost my luggage again, this is the worst service ever", 0),
    ("Feeling sick and tired, I hate Mondays so much", 0),
    ("My internet is down again and nobody answers, so frustrating and annoying", 0),
    ("RT so sad to hear the news, my heart is broken", 0),
    # difficult cases
    ("This is not good at all", 0),
    ("I don't hate it, actually it's pretty great", 1),
    ("never been happier in my life", 1),
    ("Oh great, another flat tire. Just what I needed today.", 0),
]

texts = [t for t, _ in custom_tests]
truth = np.array([l for _, l in custom_tests])
vec = tfidf.transform([clean_tweet(t) for t in texts])
pred = model.predict(vec)
proba = model.predict_proba(vec)

custom_df = pd.DataFrame({
    "Tweet": [t[:62] + ("..." if len(t) > 62 else "") for t in texts],
    "Expected": ["Positive" if l else "Negative" for l in truth],
    "Predicted": ["Positive" if p else "Negative" for p in pred],
    "Confidence": [f"{max(p) * 100:.1f}%" for p in proba],
    "Correct": ["yes" if a == b else "NO" for a, b in zip(truth, pred)],
})
display(custom_df)

print(f"Accuracy on the 8 clear tweets      : {(truth[:8] == pred[:8]).mean() * 100:.1f}%")
print(f"Accuracy on the 4 difficult tweets  : {(truth[8:] == pred[8:]).mean() * 100:.1f}%")
print(f"Accuracy on all 12 custom tweets    : {(truth == pred).mean() * 100:.1f}%")
print("\nNote: sarcasm and negation are hard for a bag-of-words model. Stop-word removal also deletes")
print("'not', 'no' and 'never', which explains some of the difficult-case mistakes.")

In [ ]:
def predict_tweet(tweet):
    """Clean one tweet, predict its sentiment and show the confidence."""
    p = model.predict_proba(tfidf.transform([clean_tweet(tweet)]))[0]
    label = "Positive" if p[1] >= 0.5 else "Negative"
    print(f"{label:<9} ({max(p) * 100:5.1f}% sure)  ->  {tweet}")

predict_tweet("Loving the new update, works great!")
predict_tweet("Worst customer support I have ever seen")

## Test 5: most informative words

In [ ]:
coefficients = model.coef_[0]
top_pos = np.argsort(coefficients)[-15:]
top_neg = np.argsort(coefficients)[:15]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(feature_names[top_pos], coefficients[top_pos], color="green")
axes[0].set_title("Terms that push towards POSITIVE")
axes[1].barh(feature_names[top_neg], coefficients[top_neg], color="red")
axes[1].set_title("Terms that push towards NEGATIVE")
plt.tight_layout()
plt.show()

## Test 6: error analysis

In [ ]:
wrong = np.where(~correct)[0]
print(f"Misclassified tweets: {len(wrong):,} of {len(y_test):,} ({len(wrong) / len(y_test) * 100:.1f}%)")

confident_wrong = wrong[np.argsort(-confidence[wrong])[:8]]
rows = []
for i in confident_wrong:
    rows.append({
        "Tweet": X_test.iloc[i][:80],
        "Actual": "Positive" if y_test.iloc[i] == 1 else "Negative",
        "Predicted": "Positive" if y_pred[i] == 1 else "Negative",
        "Confidence": f"{confidence[i] * 100:.1f}%",
    })
display(pd.DataFrame(rows))
print("Many of these are sarcastic, ambiguous, or even look wrongly labelled in the dataset (labels came from emoticons).")

## Conclusion

Tweets were cleaned, converted to TF-IDF features and classified as positive or negative with Logistic Regression
(about 78 % accuracy on 319,000 unseen tweets). The testing section verified the cleaning function with unit
tests, compared three models with ROC curves, showed that confident predictions are more accurate, tested
hand-written tweets (including negation and sarcasm) and studied the model's mistakes.